# Comparability for Origin-Destination Survey Zones in São Paulo Metropolitan Region (RMSP)

In [1]:
from utils import *

In [2]:
BASENAME = 'Metro_OD' # Nome da compatibilização
location = 'OD_RMSP'

## 1. OD layers pre-processing
The following cell pre-processes the census layers and only needs to run once

In [3]:
# Criação das malhas
od_id = {
    1987: 'Zona87',
    1997: 'Zona97',
    2007: 'Zona07',
    2017: 'NumeroZona',
    2023: 'NumeroZona',
}

malhas = {}
for k, v in od_id.items():
    malhas[k] = makeCensusLayer('resources/metrosp/Zonas OD RMSP.gpkg',
                                id_column='GEOID', 
                                layer=f'{k}',
                                len_higher_hierarchy=2,
                                geosys='SIRGAS2000')
    malhas[k].to_parquet(f'results/{BASENAME}_{k}.parquet')

'makeCensusLayer' executed in 0.7013s.
'makeCensusLayer' executed in 1.0218s.
'makeCensusLayer' executed in 1.2194s.
'makeCensusLayer' executed in 1.4275s.
'makeCensusLayer' executed in 1.3621s.


## 2. Recursive comparability graph

In [4]:
list_malhas = [(k,v) for k, v in malhas.items()]

def compatRoutine(a, b):
    n1, m1 = a
    n2, m2 = b
    print(f'Starting {n1}-{n2}...')
    G_compat = comparability_graph(m1, m2)
    G_compat.findMaintenances()
    G_compat.findSplitings(threshold=0.8)

    # Aplica séries de buffers
    for b in [-100, -50, 0]:
        G_compat.forceOverlay(buffer=b)
    # Export
    G_compat.exportCompatFiles(BASENAME, n1, n2)
    # Make coverage from export
    m12 = makeCensusLayer(f'results/{BASENAME}_MCA.gpkg',
                            layer=f'{n1}-{n2}',
                            id_column='CD_MCA',
                            len_higher_hierarchy=2,
                            is_utm=True)
    print(f'{n1}-{n2} ok!\n')
    return (f'{n1}-{n2}', m12)

def recursiveCompat(coverage_list):
    if len(coverage_list)==2:
        m = compatRoutine(coverage_list[0], coverage_list[1])
        return m
    else:
        m = compatRoutine(coverage_list[0], recursiveCompat(coverage_list[1:]))
        return m

m = recursiveCompat(list_malhas)

Starting 2017-2023...
'comparability_graph.makeGridUnion' executed in 7.6538s.
'getNeighborhoods' executed in 0.0000s.
'getNeighborhoods' executed in 0.0000s.
'comparability_graph.classifyGridChanges' executed in 0.1370s.
'comparability_graph.__init__' executed in 7.8908s.
'comparability_graph.findMaintenances' executed in 0.0000s.
'comparability_graph.findSplitings' executed in 0.0324s.
'comparability_graph.forceOverlay' executed in 0.0067s.
'comparability_graph.forceOverlay' executed in 0.0081s.
'comparability_graph.forceOverlay' executed in 0.0064s.
'comparability_graph.exportCompatFiles' executed in 0.3597s.
'makeCensusLayer' executed in 1.2007s.
2017-2023 ok!

Starting 2007-2017-2023...
'comparability_graph.makeGridUnion' executed in 10.8038s.
'getNeighborhoods' executed in 0.0000s.
'getNeighborhoods' executed in 0.0000s.
'comparability_graph.classifyGridChanges' executed in 0.0110s.
'comparability_graph.__init__' executed in 10.8889s.
'comparability_graph.findMaintenances' execut